# Aegis — End-to-End Demo Notebook
### Self-Evolving Multi-Agent System (Capstone Project)
This notebook demonstrates the complete mission flow for Aegis:
- Mission → Planner  
- Planner → Orchestrator  
- Orchestrator → Specialist Agents  
- Session + Trace logging  
- Judge scoring  
- AgentCreator auto-evolving new agents when coverage is low  

This notebook runs **entirely offline** using mocked agents — no API keys required.

In [ ]:
!pip install -q rich
import sys, os

## Load the Aegis repository
Upload your ZIP in Kaggle → Add Data → Upload File → Aegis-Self-Evolving-Agent-Fleet-main.zip

In [ ]:
import zipfile

zip_path = "/mnt/data/Aegis-Self-Evolving-Agent-Fleet-main.zip"
extract_dir = "/kaggle/working/aegis_repo"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Extracted repo to:", extract_dir)

In [ ]:
repo_root = "/kaggle/working/aegis_repo/Aegis-Self-Evolving-Agent-Fleet-main"
sys.path.append(repo_root)

print("Python path updated:", repo_root)
os.listdir(repo_root)

## Importing Aegis Components

In [ ]:
from services.orchestrator.orchestrator import Orchestrator
from services.agents.market_research_agent import MarketResearchAgent
from services.agents.copy_agent import CopyAgent
from services.agents.webdev_agent import WebDevAgent

orch = Orchestrator()

orch.register_agent("MarketResearchAgent", MarketResearchAgent(), "research")
orch.register_agent("CopyAgent", CopyAgent(), "copy")
orch.register_agent("WebDevAgent", WebDevAgent(), "deploy")

print("Registered agents:", orch.registry.list())

## Run a mission through the Aegis pipeline

In [ ]:
mission = "Launch a micro marketing campaign for a new smartwatch for college students in Bangalore."

out = orch.run_mission(mission)

import json
print(json.dumps({
    "score": out["score"],
    "session_id": out["session_id"],
    "steps": list(out["results"].keys()),
    "trace_length": len(out["trace"])
}, indent=2))

## Trace

In [ ]:
from rich.pretty import pprint
pprint(out["trace"])

## Session Events

In [ ]:
session = orch.session_service.get_or_create(out["session_id"])
from rich.pretty import pprint
pprint(session.events)

## Auto-evolution & Re-run

In [ ]:
if out["score"] < 0.80:
    print("Score < 0.80 → AgentCreator triggered an auto-generated agent.")
    print("Registry:", orch.registry.list())

    out2 = orch.run_mission(mission)

    print("\nRe-run summary:")
    print(json.dumps({
        "score": out2["score"],
        "steps": list(out2["results"].keys()),
        "trace_length": len(out2["trace"])
    }, indent=2))

    print("\nTrace after evolution:")
    pprint(out2["trace"])
else:
    print("Score high enough → No evolution required.")

# ✔️ Demo Complete
This notebook demonstrated:
- Mission → Planner → Agents → Trace → Judge  
- Session memory  
- Autonomic improvement via AgentCreator  
- Clear, reproducible offline execution